# 🦇 Batman Box Office Analysis (1966–2022)
**A data science exploration of every theatrical Batman film — budgets, revenues, ROI, critical scores, and era comparisons.**

Data sourced from Box Office Mojo, The Numbers, and Rotten Tomatoes.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='darkgrid')

# Era colour palette
ERA_COLORS = {
    'Classic':    '#F5C518',
    'Burton':     '#6A0DAD',
    'Animated':   '#1DB954',
    'Schumacher': '#FF4500',
    'Nolan':      '#1E90FF',
    'DCEU':       '#C0392B',
    'Reeves':     '#2C3E50'
}

print('✅ Libraries loaded successfully')

## 2. Load & Explore the Data

In [ ]:
df = pd.read_csv('batman_data.csv')

# Calculate key financial metrics
df['roi'] = ((df['worldwide_gross_m'] - df['budget_m']) / df['budget_m'] * 100).round(1)
df['profit_m'] = (df['worldwide_gross_m'] - df['budget_m']).round(1)
df['score_gap'] = df['rt_score'] - df['audience_score']

print(f'Total films: {len(df)}')
print(f'Years covered: {df["year"].min()} – {df["year"].max()}')
print(f'Total franchise gross: ${df["worldwide_gross_m"].sum():.0f}M')
print(f'Total franchise budget: ${df["budget_m"].sum():.0f}M')
print(f'Total franchise profit: ${df["profit_m"].sum():.0f}M')
print()
df[['title', 'year', 'budget_m', 'worldwide_gross_m', 'profit_m', 'roi', 'rt_score']].style.background_gradient(subset=['roi'], cmap='RdYlGn')

## 3. Worldwide Gross by Film

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

colors = [ERA_COLORS[era] for era in df['era']]
bars = ax.bar(df['title'], df['worldwide_gross_m'], color=colors, edgecolor='white', linewidth=0.8)

# Add value labels on bars
for bar, val in zip(bars, df['worldwide_gross_m']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'${val:.0f}M', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Batman Movies — Worldwide Box Office Gross (1966–2022)', fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Worldwide Gross ($M)', fontsize=12)
ax.set_xlabel('')
ax.set_xticklabels(df['title'], rotation=35, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

# Legend for eras
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=era) for era, color in ERA_COLORS.items()]
ax.legend(handles=legend_elements, title='Era', loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('01_worldwide_gross.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as 01_worldwide_gross.png')

## 4. ROI (Return on Investment) by Film

In [ ]:
df_sorted = df.sort_values('roi', ascending=True)
colors_sorted = [ERA_COLORS[era] for era in df_sorted['era']]
bar_colors = ['#e74c3c' if v < 0 else c for v, c in zip(df_sorted['roi'], colors_sorted)]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(df_sorted['title'], df_sorted['roi'], color=bar_colors, edgecolor='white', linewidth=0.8)

ax.axvline(0, color='black', linewidth=1.2, linestyle='--')

for bar, val in zip(bars, df_sorted['roi']):
    offset = 5 if val >= 0 else -5
    align = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', ha=align, fontsize=9, fontweight='bold')

ax.set_title('Batman Movies — Return on Investment (ROI %)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('ROI (%)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.savefig('02_roi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as 02_roi.png')

## 5. Budget vs Worldwide Gross — Are Bigger Budgets Worth It?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for _, row in df.iterrows():
    ax.scatter(row['budget_m'], row['worldwide_gross_m'],
               color=ERA_COLORS[row['era']], s=180, zorder=5,
               edgecolors='white', linewidth=1.5)
    ax.annotate(row['title'],
                (row['budget_m'], row['worldwide_gross_m']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8)

# Break-even line (gross = budget)
max_val = max(df['budget_m'].max(), df['worldwide_gross_m'].max()) + 50
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, alpha=0.4, label='Break-even line')

# Trend line
slope, intercept, r, p, _ = stats.linregress(df['budget_m'], df['worldwide_gross_m'])
x_line = np.linspace(df['budget_m'].min(), df['budget_m'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=1.5,
        alpha=0.6, label=f'Trend line (R²={r**2:.2f})')

ax.set_title('Budget vs Worldwide Gross — Does Spending More Pay Off?', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Production Budget ($M)', fontsize=12)
ax.set_ylabel('Worldwide Gross ($M)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

legend_elements = [Patch(facecolor=color, label=era) for era, color in ERA_COLORS.items()]
legend_elements += [plt.Line2D([0],[0], linestyle='--', color='black', alpha=0.4, label='Break-even'),
                    plt.Line2D([0],[0], linestyle='-', color='red', alpha=0.6, label=f'Trend (R²={r**2:.2f})')]
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('03_budget_vs_gross.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation R² = {r**2:.2f} | slope = {slope:.2f} (every $1M budget → ${slope:.1f}M gross)')

## 6. Critics vs Audience — The Mask of the Phantasm Anomaly

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for _, row in df.iterrows():
    size = row['worldwide_gross_m'] * 0.5  # bubble size = gross
    ax.scatter(row['rt_score'], row['worldwide_gross_m'],
               color=ERA_COLORS[row['era']], s=size,
               alpha=0.8, edgecolors='white', linewidth=1.5, zorder=5)
    ax.annotate(row['title'],
                (row['rt_score'], row['worldwide_gross_m']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

# Trend line
slope, intercept, r, p, _ = stats.linregress(df['rt_score'], df['worldwide_gross_m'])
x_line = np.linspace(df['rt_score'].min(), df['rt_score'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=1.5, alpha=0.6,
        label=f'Trend line (R²={r**2:.2f})')

ax.set_title('Rotten Tomatoes Score vs Worldwide Gross\n(Bubble size = box office gross)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Rotten Tomatoes Score (%)', fontsize=12)
ax.set_ylabel('Worldwide Gross ($M)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

legend_elements = [Patch(facecolor=color, label=era) for era, color in ERA_COLORS.items()]
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('04_score_vs_gross.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight the anomaly
phantasm = df[df['title'] == 'Batman: Mask of the Phantasm'].iloc[0]
print(f"\n🔍 ANOMALY: Mask of the Phantasm")
print(f"   RT Score: {phantasm['rt_score']}% (one of the highest)")
print(f"   Worldwide Gross: ${phantasm['worldwide_gross_m']}M (the lowest of the modern era)")
print(f"   Reason: rushed to theaters in 8 months, almost no marketing campaign")

## 7. Era Comparison — Which Batman Era Was Most Successful?

In [ ]:
era_stats = df.groupby('era').agg(
    films=('title', 'count'),
    avg_gross=('worldwide_gross_m', 'mean'),
    avg_roi=('roi', 'mean'),
    avg_rt=('rt_score', 'mean'),
    total_gross=('worldwide_gross_m', 'sum')
).round(1).reset_index()

era_order = ['Classic', 'Burton', 'Animated', 'Schumacher', 'Nolan', 'DCEU', 'Reeves']
era_stats['era'] = pd.Categorical(era_stats['era'], categories=era_order, ordered=True)
era_stats = era_stats.sort_values('era')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
era_colors = [ERA_COLORS[e] for e in era_stats['era']]

# Average gross
axes[0].bar(era_stats['era'], era_stats['avg_gross'], color=era_colors, edgecolor='white')
axes[0].set_title('Avg Worldwide Gross per Film', fontweight='bold')
axes[0].set_ylabel('$M')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(era_stats['avg_gross']):
    axes[0].text(i, v + 10, f'${v:.0f}M', ha='center', fontsize=9, fontweight='bold')

# Average ROI
bar_colors_roi = ['#e74c3c' if v < 0 else ERA_COLORS[e] for v, e in zip(era_stats['avg_roi'], era_stats['era'])]
axes[1].bar(era_stats['era'], era_stats['avg_roi'], color=bar_colors_roi, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('Avg ROI per Era', fontweight='bold')
axes[1].set_ylabel('ROI %')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(era_stats['avg_roi']):
    axes[1].text(i, v + 5, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')

# Average RT score
axes[2].bar(era_stats['era'], era_stats['avg_rt'], color=era_colors, edgecolor='white')
axes[2].set_title('Avg Rotten Tomatoes Score per Era', fontweight='bold')
axes[2].set_ylabel('RT Score (%)')
axes[2].set_ylim(0, 110)
axes[2].tick_params(axis='x', rotation=30)
for i, v in enumerate(era_stats['avg_rt']):
    axes[2].text(i, v + 2, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Batman Era Comparison — Gross, ROI, and Critical Reception', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('05_era_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEra Summary Table:')
print(era_stats.to_string(index=False))

## 8. Budget Inflation Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

colors = [ERA_COLORS[era] for era in df['era']]

ax.fill_between(df['year'], df['budget_m'], alpha=0.15, color='royalblue')
ax.plot(df['year'], df['budget_m'], 'o-', color='royalblue', linewidth=2,
        markersize=8, label='Budget', zorder=5)
ax.plot(df['year'], df['worldwide_gross_m'], 's--', color='darkorange', linewidth=2,
        markersize=8, label='Worldwide Gross', zorder=5)

for _, row in df.iterrows():
    ax.annotate(row['title'].replace('Batman: ', '').replace('Batman ', ''),
                (row['year'], row['budget_m']),
                textcoords='offset points', xytext=(0, -18),
                fontsize=7, ha='center', color='royalblue', rotation=20)

ax.set_title('Batman Franchise — Budget vs Gross Over Time (1966–2022)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Amount ($M)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('06_budget_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Budget increase from 1989 to 2012: ${df[df['year']==1989]['budget_m'].values[0]}M → ${df[df['year']==2012]['budget_m'].values[0]}M")
print(f"That's a {((250-35)/35*100):.0f}% increase over 23 years")

## 9. Animated vs Live Action Comparison

In [ ]:
anim_vs_live = df.groupby('animated').agg(
    films=('title', 'count'),
    avg_budget=('budget_m', 'mean'),
    avg_gross=('worldwide_gross_m', 'mean'),
    avg_roi=('roi', 'mean'),
    avg_rt=('rt_score', 'mean')
).round(1)
anim_vs_live.index = ['Live Action', 'Animated']

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

categories = ['Avg Budget ($M)', 'Avg Gross ($M)', 'Avg ROI (%)', 'Avg RT Score (%)']
live_vals = [anim_vs_live.loc['Live Action', 'avg_budget'],
             anim_vs_live.loc['Live Action', 'avg_gross'],
             anim_vs_live.loc['Live Action', 'avg_roi'],
             anim_vs_live.loc['Live Action', 'avg_rt']]
anim_vals = [anim_vs_live.loc['Animated', 'avg_budget'],
             anim_vs_live.loc['Animated', 'avg_gross'],
             anim_vs_live.loc['Animated', 'avg_roi'],
             anim_vs_live.loc['Animated', 'avg_rt']]

x = np.arange(len(categories))
width = 0.35
axes[0].bar(x - width/2, live_vals, width, label='Live Action', color='#1E90FF', edgecolor='white')
axes[0].bar(x + width/2, anim_vals, width, label='Animated', color='#1DB954', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, rotation=20, ha='right', fontsize=9)
axes[0].set_title('Animated vs Live Action — Key Metrics', fontweight='bold')
axes[0].legend()

# Scatter: RT score vs ROI coloured by type
for _, row in df.iterrows():
    color = '#1DB954' if row['animated'] == 'Yes' else '#1E90FF'
    label = 'Animated' if row['animated'] == 'Yes' else 'Live Action'
    axes[1].scatter(row['rt_score'], row['roi'], color=color, s=120,
                    edgecolors='white', linewidth=1.2, zorder=5)
    axes[1].annotate(row['title'].replace('Batman: ', ''),
                     (row['rt_score'], row['roi']),
                     textcoords='offset points', xytext=(5, 3), fontsize=7)

axes[1].axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
axes[1].set_title('RT Score vs ROI — Animated vs Live Action', fontweight='bold')
axes[1].set_xlabel('RT Score (%)')
axes[1].set_ylabel('ROI (%)')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(facecolor='#1DB954', label='Animated'),
                         Patch(facecolor='#1E90FF', label='Live Action')])

plt.suptitle('Animated vs Live Action Batman Films', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_animated_vs_live.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAnimated vs Live Action Summary:')
print(anim_vs_live)

## 10. Key Findings & Conclusions

In [ ]:
best_roi    = df.loc[df['roi'].idxmax()]
worst_roi   = df.loc[df['roi'].idxmin()]
highest_rt  = df.loc[df['rt_score'].idxmax()]
lowest_rt   = df.loc[df['rt_score'].idxmin()]
top_gross   = df.loc[df['worldwide_gross_m'].idxmax()]
phantasm    = df[df['title'] == 'Batman: Mask of the Phantasm'].iloc[0]

print('=' * 60)
print('🦇 BATMAN BOX OFFICE ANALYSIS — KEY FINDINGS')
print('=' * 60)
print(f"\n💰 Highest Grossing:    {top_gross['title']} ({top_gross['year']}) — ${top_gross['worldwide_gross_m']}M")
print(f"📈 Best ROI:            {best_roi['title']} ({best_roi['year']}) — {best_roi['roi']}%")
print(f"📉 Worst ROI:           {worst_roi['title']} ({worst_roi['year']}) — {worst_roi['roi']}%")
print(f"⭐ Highest RT Score:    {highest_rt['title']} ({highest_rt['year']}) — {highest_rt['rt_score']}%")
print(f"💀 Lowest RT Score:     {lowest_rt['title']} ({lowest_rt['year']}) — {lowest_rt['rt_score']}%")
print(f"\n🔍 ANOMALY — Mask of the Phantasm ({phantasm['year']}):")
print(f"   RT Score {phantasm['rt_score']}% (top 3 all-time) but only ${phantasm['worldwide_gross_m']}M gross")
print(f"   Reason: rushed 8-month production, minimal marketing, poor theatrical rollout")
print(f"   Lesson: quality alone does not guarantee box office success")
print(f"\n🏆 Best Era (by avg gross): Nolan Trilogy — avg ${df[df['era']=='Nolan']['worldwide_gross_m'].mean():.0f}M per film")
print(f"🎨 Best Era (by avg RT):   Nolan Trilogy — avg {df[df['era']=='Nolan']['rt_score'].mean():.0f}% RT score")
print(f"\n💡 Correlation (budget vs gross): R² = {stats.linregress(df['budget_m'], df['worldwide_gross_m'])[2]**2:.2f}")
print(f"   Higher budgets are moderately correlated with higher gross, but not guaranteed")
print('=' * 60)